In [ ]:
import zipfile
import os
import psutil
from tqdm import tqdm
import pandas as pd
import pickle

In [ ]:
data_file_path = '/users/xwang259/scratch/NRD/NRD_2021_Core.csv'
label_path = '/users/xwang259/scratch/NRD/nrd2021.dta'
index_path = '/users/xwang259/scratch/NRD/NRD_2021_comorbidity_index_Shu.dta'

In [ ]:
# Read the entire CSV in one go
label_df = pd.read_stata(
    label_path,
)

# Quick sanity check
print(label_df.shape)
print(label_df.head())

In [ ]:
counts = label_df['missing'].value_counts()
print(counts)

In [ ]:
subset = label_df[label_df['missing'] == 1]
print(subset.head())

In [ ]:
# Read the entire CSV in one go
index_df = pd.read_stata(
    index_path,
)

# Quick sanity check
print(index_df.shape)
print(index_df.head())

In [ ]:
# Read the entire CSV in one go
df = pd.read_csv(data_file_path, header=None)


# # Read the entire CSV in one go
# data = pd.read_stata(
#     data_file_path,
# )

In [ ]:
# Quick sanity check
print(df.shape)
print(df.head())

In [ ]:
columns = [
'age','aweekend','died','discwt','dispuniform','dmonth','dqtr','drg','drgver','drg_nopoa','elective','female',
'hcup_ed','hosp_nrd','i10_birth','i10_delivery','i10_dx1','i10_dx2','i10_dx3','i10_dx4','i10_dx5','i10_dx6',
'i10_dx7','i10_dx8','i10_dx9','i10_dx10','i10_dx11','i10_dx12','i10_dx13','i10_dx14','i10_dx15','i10_dx16',
'i10_dx17','i10_dx18','i10_dx19','i10_dx20','i10_dx21','i10_dx22','i10_dx23','i10_dx24','i10_dx25','i10_dx26',
'i10_dx27','i10_dx28','i10_dx29','i10_dx30','i10_dx31','i10_dx32','i10_dx33','i10_dx34','i10_dx35','i10_dx36',
'i10_dx37','i10_dx38','i10_dx39','i10_dx40','i10_injury','i10_multinjury','i10_ndx','i10_npr',
'i10_pr1','i10_pr2','i10_pr3','i10_pr4','i10_pr5','i10_pr6','i10_pr7','i10_pr8','i10_pr9','i10_pr10','i10_pr11',
'i10_pr12','i10_pr13','i10_pr14','i10_pr15','i10_pr16','i10_pr17','i10_pr18','i10_pr19','i10_pr20','i10_pr21',
'i10_pr22','i10_pr23','i10_pr24','i10_pr25','i10_serviceline','key_nrd','los','mdc','mdc_nopoa',
'nrd_daystoevent','nrd_stratum','nrd_visitlink','pay1','pclass_orproc','pl_nchs','prday1','prday2','prday3',
'prday4','prday5','prday6','prday7','prday8','prday9','prday10','prday11','prday12','prday13','prday14',
'prday15','prday16','prday17','prday18','prday19','prday20','prday21','prday22','prday23','prday24',
'prday25','rehabtransfer','resident','samedayevent','totchg','year','zipinc_qrtl'
]4

In [ ]:
df.columns = columns

In [ ]:
print(list(df.columns))     # shows as a python list

In [ ]:
# method 1: chain .merge()
merged = (
    df
    .merge(index_df, on="key_nrd", how="inner")   # “inner” keeps only keys present in both
    .merge(label_df, on="key_nrd", how="inner")
)

In [ ]:
desired_columns = ['age', 'died', 'i10_dx1', 'i10_dx2', 'i10_dx3', 'i10_dx4', 'i10_dx5', 'i10_dx6', 'i10_dx7', 'i10_dx8', 'i10_dx9', 
                   'i10_dx10', 'i10_dx11', 'i10_dx12', 'i10_dx13', 'i10_dx14', 'i10_dx15', 'i10_dx16', 'i10_dx17', 'i10_dx18', 'i10_dx19', 
                   'i10_dx20', 'i10_dx21', 'i10_dx22', 'i10_dx23', 'i10_dx24', 'i10_dx25', 'i10_dx26', 'i10_dx27', 'i10_dx28', 'i10_dx29', 
                   'i10_dx30', 'i10_dx31', 'i10_dx32', 'i10_dx33', 'i10_dx34', 'i10_dx35', 'i10_dx36', 'i10_dx37', 'i10_dx38', 'i10_dx39', 
                   'i10_dx40', 'female', 'key_nrd', 'pay1', 'year', 'zipinc_qrtl', 
                   'missing', 'mor30', 'rea30', 'charlindex', 'age_adjust', 'charlindex_age_adjust', 'Index_Readmission', 'Index_Mortality', '_merge']

In [ ]:
# ✅ Keep only columns that exist in both desired list & current file
available_cols = [c for c in desired_columns if c in merged.columns]
print(available_cols)

In [ ]:
# ✅ Make an explicit copy before modifying
merged = merged[available_cols].copy()        
# ✅ Ensure all desired columns exist (fill missing ones with NaN)
for col in desired_columns:
    if col not in merged.columns:
        merged[col] = pd.NA
# ✅ Reorder columns consistently
merged = merged[desired_columns]

In [ ]:
# 1) CSV
merged.to_csv("/users/xwang259/scratch/NRD/NRD_2021_test.csv", index=False)

In [ ]:
### TEST
file_2021 = '/users/xwang259/scratch/NRD/NRD_2021_test.csv'
df1 = pd.read_csv(file_2021)
df1['pay1']

In [ ]:
df1.head(100).to_csv("first_100_rows_2021.csv", index=False)

In [ ]:
### TEST
file_2022 = '/users/xwang259/scratch/NRD/NRD_2022_test.csv'
df2 = pd.read_csv(file_2022)
df2['pay1']

In [ ]:
df2.head(100).to_csv("first_100_rows_2022.csv", index=False)

In [ ]:
# Check for invalid PAY1 values
print("\n=== INSPECTING PAY1 VALUES ===")
print(f"Unique PAY1 values in new_data: {sorted(df1['pay1'].unique())}")
print(f"Value counts for PAY1:")
print(df1['pay1'].value_counts().sort_index())

In [ ]:
# Check for invalid PAY1 values
print("\n=== INSPECTING PAY1 VALUES ===")
print(f"Unique PAY1 values in new_data: {sorted(df1['age'].unique())}")
print(f"Value counts for PAY1:")
print(df1['age'].value_counts().sort_index())

In [ ]:
# Check for invalid PAY1 values
print("\n=== INSPECTING ZIPINC VALUES ===")
print(f"Unique ZIPINC values in new_data: {sorted(df1['zipinc_qrtl'].unique())}")
print(f"Value counts for ZIPINC:")
print(df1['pay1'].value_counts().sort_index())

In [ ]:
print(f"2021 - PAY1 null count: {df1['pay1'].isna().sum()}")

In [ ]:
# Identify rows with negative or unusual PAY1 values
invalid_pay1_mask = df1['pay1'] < 0  # or whatever defines "invalid" for your use case
print(f"\nNumber of rows with PAY1 < 0: {invalid_pay1_mask.sum()}")


In [ ]:
if invalid_pay1_mask.sum() > 0:
    print("\nSample rows with invalid PAY1 values:")
    print(df1[invalid_pay1_mask][['pay1', 'zipinc_qrtl', 'age', 'female', 'died', 'mor30', 'rea30']].head(20))
    
    # Get all unique invalid values
    invalid_values = new_data.loc[invalid_pay1_mask, 'pay1'].unique()
    print(f"\nAll invalid PAY1 values found: {sorted(invalid_values)}")
    
    # Count by value
    print("\nBreakdown of invalid values:")
    print(df1.loc[invalid_pay1_mask, 'pay1'].value_counts())

In [ ]:
# Check for invalid PAY1 values
print("\n=== INSPECTING PAY1 VALUES ===")
print(f"Unique PAY1 values in new_data: {sorted(df2['pay1'].unique())}")
print(f"Value counts for PAY1:")
print(df2['pay1'].value_counts().sort_index())

In [ ]:
# Check for invalid PAY1 values
print("\n=== INSPECTING PAY1 VALUES ===")
print(f"Unique PAY1 values in new_data: {sorted(df2['zipinc_qrtl'].unique())}")
print(f"Value counts for PAY1:")
print(df2['zipinc_qrtl'].value_counts().sort_index())

In [ ]:
# Check for invalid PAY1 values
print("\n=== INSPECTING PAY1 VALUES ===")
print(f"Unique PAY1 values in new_data: {sorted(df2['age'].unique())}")
print(f"Value counts for PAY1:")
print(df2['age'].value_counts().sort_index())

In [ ]:
X_validate = pd.read_csv('/users/xwang259/icd/data/mort_nodie/X_test.csv')

In [ ]:
# Quick sanity check
print(X_validate.shape)
print(X_validate.head())
print(X_validate.columns)

In [ ]:
X_validate.head(100).to_csv("first_100_rows_X_validate.csv", index=False)

In [ ]:
### Finished

In [ ]:
# Function to print memory usage
def print_memory_usage():
    process = psutil.Process()
    print(f"Memory usage: {process.memory_info().rss / 1e6} MB")

In [ ]:
desired_columns = ['age', 'died', 'i10_dx1', 'i10_dx2', 'i10_dx3', 'i10_dx4', 'i10_dx5', 'i10_dx6', 'i10_dx7', 'i10_dx8', 'i10_dx9', 
                   'i10_dx10', 'i10_dx11', 'i10_dx12', 'i10_dx13', 'i10_dx14', 'i10_dx15', 'i10_dx16', 'i10_dx17', 'i10_dx18', 'i10_dx19', 
                   'i10_dx20', 'i10_dx21', 'i10_dx22', 'i10_dx23', 'i10_dx24', 'i10_dx25', 'i10_dx26', 'i10_dx27', 'i10_dx28', 'i10_dx29', 
                   'i10_dx30', 'i10_dx31', 'i10_dx32', 'i10_dx33', 'i10_dx34', 'i10_dx35', 'i10_dx36', 'i10_dx37', 'i10_dx38', 'i10_dx39', 
                   'i10_dx40', 'female', 'key_nrd', 'pay1', 'year', 'zipinc_qrtl', 
                   'missing', 'mor30', 'rea30', 'charlindex', 'age_adjust', 'charlindex_age_adjust', 'Index_Readmission', 'Index_Mortality', '_merge']

In [ ]:
file_paths = [
    '/users/xwang259/scratch/NRD/NRD_2016.dta',
    '/users/xwang259/scratch/NRD/NRD_2017.dta',
    '/users/xwang259/scratch/NRD/NRD_2018.dta',
    '/users/xwang259/scratch/NRD/NRD_2019.dta',
    '/users/xwang259/scratch/NRD/NRD_2020.dta'
]

output_csv = "/users/xwang259/scratch/NRD/NRD_2016_2020_new.csv"

In [ ]:
# To track whether we’ve written headers already
first_write = True

for file in tqdm(file_paths):
    print(f"Processing {file}...")
    chunk_iterator = pd.read_stata(file, chunksize=1000000)  # smaller chunks
    
    for chunk in tqdm(chunk_iterator):
        # ✅ Keep only columns that exist in both desired list & current file
        available_cols = [c for c in desired_columns if c in chunk.columns]
        # ✅ Make an explicit copy before modifying
        chunk = chunk[available_cols].copy()        
        # ✅ Ensure all desired columns exist (fill missing ones with NaN)
        for col in desired_columns:
            if col not in chunk.columns:
                chunk[col] = pd.NA
        # ✅ Reorder columns consistently
        chunk = chunk[desired_columns]
        
        # Append to CSV
        chunk.to_csv(
            output_csv,
            mode='a',                 # append mode
            header=first_write,       # write header only for first chunk
            index=False
        )
        
        if first_write:
            first_write = False  # headers only once

In [ ]:
output_csv = "/users/xwang259/scratch/NRD/NRD_2016_2020.csv"
icd_columns = [f"i10_dx{i}" for i in range(1, 41)]
# We also need AGE column in this pass
columns_to_read = icd_columns + ["age"]

unique_icd_codes = set()
age_min, age_max = float("inf"), -float("inf")

# Stream in chunks
chunk_iter = pd.read_csv(
    output_csv,
    usecols=columns_to_read,
    chunksize=500_000,
    dtype=str  # read everything as string to avoid dtype issues
)

for i, chunk in enumerate(chunk_iter):
    # === 1) Update ICD vocabulary ===
    icd_series = pd.Series(chunk[icd_columns].values.ravel())
    icd_series = icd_series.str.upper()
    unique_icd_codes.update(icd_series.unique())

    # === 2) Update AGE min/max ===
    # Convert AGE to numeric (coerce errors → NaN), drop missing
    age_vals = pd.to_numeric(chunk["age"], errors="coerce").dropna()
    if not age_vals.empty:
        age_min = min(age_min, age_vals.min())
        age_max = max(age_max, age_vals.max())

    if i % 10 == 0:
        print(f"Processed {i} chunks... ICD vocab size: {len(unique_icd_codes)}")
        break

print(f"\n✅ Finished scanning all chunks!")
print(f"Total unique ICD codes: {len(unique_icd_codes)}")
print(f"AGE range: {age_min} - {age_max}")

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
unique_icd_codes = [("NAN" if str(x) == "nan" else str(x)) for x in unique_icd_codes]
encoder.fit(list(unique_icd_codes))
print("Num unique ICD codes:", len(encoder.classes_))

In [ ]:
print(encoder.classes_)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

age_scaler = MinMaxScaler(feature_range=(0,1))
age_scaler.min_ = 0.0
age_scaler.scale_ = 1.0 / (age_max - age_min)  # precomputed scaling

### Processing

In [ ]:
# Load the LabelEncoder for ICD codes
with open('Model/full_label_encoder.pkl', 'rb') as file:
    encoder = pickle.load(file)

# Load the MinMaxScaler for 'AGE'
with open('Model/full_age_scaler.pkl', 'rb') as file:
    age_scaler = pickle.load(file)

In [ ]:
output_csv = "/users/xwang259/scratch/NRD/NRD_2016_2020.csv"

outcome_var = 'MOR30'  # Define the outcome variable here, e.g., 'DIED', 'MOR30', 'REA30'

# Columns we’ll process
icd_columns = [f"I10_DX{i}" for i in range(1, 41)]

X_list = []
y_list = []

chunk_iter = pd.read_csv(
    output_csv,
    chunksize=500_000,
    # dtype=str  # read everything as string for consistency
)

for i, data in enumerate(chunk_iter):
    print(f"Processing chunk {i}...")
        
    # Step 1: onvert all column names to uppercase
    data.columns = data.columns.str.upper()
    
    # Step 2: Data Preprocessing
    # Filter out observations where DIED == 1 if outcome_var is REA30
    if outcome_var == 'REA30' and 'DIED' in data.columns:
        data = data[data['DIED'] != 1]

    # Handle missing values in the target variable
    data = data.dropna(subset=[outcome_var])

    # --- 1) Normalize AGE ---
    # data['AGE'] = age_scaler.transform(data[['AGE']])
    data.loc[:, 'AGE'] = data['AGE'].astype(float)
    data.loc[:, 'AGE'] = age_scaler.transform(data[['AGE']])
    print(data['AGE'].dtype)
    
    # --- 2) Encode ICD codes ---
    for col in icd_columns:
        # data[col] = encoder.transform(data[col].astype(str).str.upper())
        data.loc[:, col] = encoder.transform(data[col].astype(str).str.upper())
        
    # --- 3) One-hot encode PAY1 + ZIPINC_QRTL ---
    data = pd.get_dummies(data, columns=['PAY1', 'ZIPINC_QRTL'], prefix=['PAY1', 'ZIPINC_QRTL'])


    # Separate features and target variable
    X_chunk = data[['AGE', 'FEMALE'] + list(data.filter(regex='PAY1_').columns) + list(data.filter(regex='ZIPINC_QRTL_').columns) + icd_columns]
    y_chunk = data[outcome_var]
    
    # Handle missing values in features (if any)
    X_chunk = X_chunk.dropna()
    y_chunk = y_chunk.loc[X_chunk.index]
    
    # ✅ Append to global list
    X_list.append(X_chunk)
    y_list.append(y_chunk) 
    
    print(f"Chunk {i} processed: X={X_chunk.shape}, y={y_chunk.shape}")

In [ ]:
# ✅ After all chunks processed → Concatenate everything
X_all = pd.concat(X_list, ignore_index=True)
y_all = pd.concat(y_list, ignore_index=True)

print("\n✅ All chunks processed & combined!")
print("Final combined shape:", X_all.shape, y_all.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (stratify to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)

# Apply SMOTE to the training data only
print("Original class distribution:", pd.Series(y_train).value_counts())

In [ ]:
from sklearn.utils import resample
from sklearn.utils import shuffle
import numpy as np

# Separate the majority and minority classes
X_majority = X_train[y_train == 0]
X_minority = X_train[y_train == 1]
y_majority = y_train[y_train == 0]
y_minority = y_train[y_train == 1]


# Downsample the majority class
X_majority_downsampled, y_majority_downsampled = resample(
    X_majority, y_majority, 
    replace=False,                  # No resampling; we want unique samples
    n_samples=len(y_minority),      # Match the minority class count
    random_state=42
)

# Combine the downsampled data
# Combine minority class with downsampled majority class
X_train_downsampled = pd.concat([X_minority, X_majority_downsampled])
y_train_downsampled = pd.concat([y_minority, y_majority_downsampled])

X_train_downsampled, y_train_downsampled = shuffle(X_train_downsampled, y_train_downsampled, random_state=42)

print("New class distribution:", pd.Series(y_train_downsampled).value_counts())

In [ ]:
# Save to a new file for further use
X_train_downsampled.to_csv("/users/xwang259/scratch/NRD/processed_data/readmit/X_train_downsampled.csv", index=False)
y_train_downsampled.to_csv("/users/xwang259/scratch/NRD/processed_data/readmit/y_train_downsampled.csv", index=False)
X_test.to_csv("/users/xwang259/scratch/NRD/processed_data/readmit/X_test.csv", index=False)
y_test.to_csv("/users/xwang259/scratch/NRD/processed_data/readmit/y_test.csv", index=False)

In [ ]:

import pickle
# Save the LabelEncoder and MinMaxScaler using pickle

# Save LabelEncoder
with open('Model/full_label_encoder.pkl', 'wb') as file:
    pickle.dump(encoder, file)

# Save MinMaxScaler for 'AGE'
with open('Model/full_age_scaler.pkl', 'wb') as file:
    pickle.dump(age_scaler, file)

In [ ]:
X_train_downsampled = pd.read_csv('/users/xwang259/scratch/NRD/readmit/X_train_downsampled.csv')
y_train_downsampled = pd.read_csv('/users/xwang259/scratch/NRD/readmit/y_train_downsampled.csv')
X_test = pd.read_csv('/users/xwang259/scratch/NRD/readmit/X_test.csv')
y_test = pd.read_csv('/users/xwang259/scratch/NRD/readmit/y_test.csv')

In [ ]:
print(len(X_train_downsampled))
print(len(y_train_downsampled))
print(len(X_test))
print(len(y_test))

In [ ]:
X_train_downsampled = pd.read_csv('/users/xwang259/scratch/NRD/mort/X_train_downsampled.csv')
y_train_downsampled = pd.read_csv('/users/xwang259/scratch/NRD/mort/y_train_downsampled.csv')
X_test = pd.read_csv('/users/xwang259/scratch/NRD/mort/X_test.csv')
y_test = pd.read_csv('/users/xwang259/scratch/NRD/mort/y_test.csv')

In [ ]:
print(len(X_train_downsampled))
print(len(y_train_downsampled))
print(len(X_test))
print(len(y_test))

In [ ]:
X_train_downsampled = pd.read_csv('/users/xwang259/scratch/NRD/mort_nodie/X_train_downsampled.csv')
y_train_downsampled = pd.read_csv('/users/xwang259/scratch/NRD/mort_nodie/y_train_downsampled.csv')
X_test = pd.read_csv('/users/xwang259/scratch/NRD/mort_nodie/X_test.csv')
y_test = pd.read_csv('/users/xwang259/scratch/NRD/mort_nodie/y_test.csv')

In [ ]:
print(len(X_train_downsampled))
print(len(y_train_downsampled))
print(len(X_test))
print(len(y_test))

In [ ]:
file_2021 = '/users/xwang259/scratch/NRD/NRD_2021/NRD_2021_test.csv'
file_2022 = '/users/xwang259/scratch/NRD/NRD_2022/NRD_2022_test.csv'

# Read and combine files
df1 = pd.read_csv(file_2021)
df2 = pd.read_csv(file_2022)

# Combine rows from both files
full_data = pd.concat([df1, df2], ignore_index=True)
# Convert all column names to uppercase
full_data.columns = full_data.columns.str.upper()
full_data

In [ ]:

# Load the LabelEncoder for ICD codes
with open('Model/full_label_encoder.pkl', 'rb') as file:
    encoder = pickle.load(file)

# Load the MinMaxScaler for 'AGE'
with open('Model/full_age_scaler.pkl', 'rb') as file:
    age_scaler = pickle.load(file)


In [ ]:
# Define the outcome variable and file path
outcome_var = 'MOR30'  # Set outcome variable here, e.g., 'DIED', 'MOR30', 'REA30'

new_data = full_data

# Handle missing values in the target variable
new_data = new_data.dropna(subset=[outcome_var])

# Define ICD columns
icd_columns = [f'I10_DX{i}' for i in range(1, 41)]

# Encode ICD codes
label_to_int = {label: idx for idx, label in enumerate(encoder.classes_)}
unknown_label_int = encoder.transform(["NAN"])[0]  # Assign unknown codes to index 0

for col in icd_columns:
    new_data[col] = new_data[col].astype(str).str.upper()  # Convert to uppercase
    new_data[col] = new_data[col].map(label_to_int).fillna(unknown_label_int).astype(int)

# Normalize 'AGE'
new_data['AGE'] = age_scaler.transform(new_data[['AGE']])

# One-hot encode 'PAY1' and 'ZIPINC_QRTL' only (excluding 'RACE')
new_data = pd.get_dummies(new_data, columns=['PAY1', 'ZIPINC_QRTL'], prefix=['PAY1', 'ZIPINC_QRTL'])

# Ensure that all expected one-hot encoded columns are present
def ensure_columns(data, expected_columns):
    for col in expected_columns:
        if col not in data.columns:
            data[col] = 0
    return data

# Define one-hot encoded columns for 'PAY1' and 'ZIPINC_QRTL' based on training data
pay1_columns = [col for col in new_data.columns if col.startswith('PAY1_')]
zipinc_qrtl_columns = [col for col in new_data.columns if col.startswith('ZIPINC_QRTL_')]

# Ensure all expected columns are present
new_data = ensure_columns(new_data, pay1_columns + zipinc_qrtl_columns)

# Drop rows with missing values
X_new = new_data[['AGE', 'FEMALE'] + pay1_columns + zipinc_qrtl_columns + icd_columns]
X_new = X_new.dropna()
new_data = new_data.loc[X_new.index]


In [ ]:
new_data

In [ ]:
import numpy as np

y = new_data[outcome_var].to_numpy()
idx_all = np.arange(len(new_data))
def stratified_pick(idx, y_sub, frac, seed):
    rng = np.random.default_rng(seed)
    picked = []
    for cls in np.unique(y_sub):
        cls_idx = idx[y_sub == cls]
        n = max(1, int(np.floor(frac * len(cls_idx))))
        pick = rng.choice(cls_idx, size=n, replace=False)
        picked.append(pick)
    return np.concatenate(picked)

idx10 = stratified_pick(idx_all, y, frac=0.10, seed=42)       # 10%
df10 = new_data.iloc[idx10].copy()

print(len(df10))

In [ ]:
# Define the outcome variable and file path
outcome_var = 'REA30'  # Set outcome variable here, e.g., 'DIED', 'MOR30', 'REA30'

new_data = full_data[full_data['DIED'] != 1]

# Handle missing values in the target variable
new_data = new_data.dropna(subset=[outcome_var])

# Define ICD columns
icd_columns = [f'I10_DX{i}' for i in range(1, 41)]

# Encode ICD codes
label_to_int = {label: idx for idx, label in enumerate(encoder.classes_)}
unknown_label_int = encoder.transform(["NAN"])[0]  # Assign unknown codes to index 0

for col in icd_columns:
    new_data[col] = new_data[col].astype(str).str.upper()  # Convert to uppercase
    new_data[col] = new_data[col].map(label_to_int).fillna(unknown_label_int).astype(int)

# Normalize 'AGE'
new_data['AGE'] = age_scaler.transform(new_data[['AGE']])

# One-hot encode 'PAY1' and 'ZIPINC_QRTL' only (excluding 'RACE')
new_data = pd.get_dummies(new_data, columns=['PAY1', 'ZIPINC_QRTL'], prefix=['PAY1', 'ZIPINC_QRTL'])

# Ensure that all expected one-hot encoded columns are present
def ensure_columns(data, expected_columns):
    for col in expected_columns:
        if col not in data.columns:
            data[col] = 0
    return data

# Define one-hot encoded columns for 'PAY1' and 'ZIPINC_QRTL' based on training data
pay1_columns = [col for col in new_data.columns if col.startswith('PAY1_')]
zipinc_qrtl_columns = [col for col in new_data.columns if col.startswith('ZIPINC_QRTL_')]

# Ensure all expected columns are present
new_data = ensure_columns(new_data, pay1_columns + zipinc_qrtl_columns)

# Drop rows with missing values
X_new = new_data[['AGE', 'FEMALE'] + pay1_columns + zipinc_qrtl_columns + icd_columns]
X_new = X_new.dropna()
new_data = new_data.loc[X_new.index]


y = new_data[outcome_var].to_numpy()
idx_all = np.arange(len(new_data))
def stratified_pick(idx, y_sub, frac, seed):
    rng = np.random.default_rng(seed)
    picked = []
    for cls in np.unique(y_sub):
        cls_idx = idx[y_sub == cls]
        n = max(1, int(np.floor(frac * len(cls_idx))))
        pick = rng.choice(cls_idx, size=n, replace=False)
        picked.append(pick)
    return np.concatenate(picked)

idx10 = stratified_pick(idx_all, y, frac=0.10, seed=42)       # 10%
df10 = new_data.iloc[idx10].copy()

print(len(df10))